# 工业实践：非等温 CSTR 灰箱模型

本 Notebook 从守恒方程开始，建立放热反应 A → B 的非等温连续搅拌釜（CSTR）动态模型，并用 Keras 3 学习机理模型遗漏的反应速率修正。示例包含机理基线、灰箱模型和纯数据模型，按完整运行轨迹拆分训练/验证/测试。

**重要边界：** 没有提供工厂数据。下文的可运行结果来自带噪声的合成数据，只用于演示方法，不代表真实装置验证或可直接用于控制、安全联锁、工艺放大或生产决策。

## 1. 机理方程、边界与单位

假设液相恒容、完全混合、单一不可逆放热反应 A → B。状态为反应物浓度 `C_A`（mol/L）和釜温 `T`（K）；操作量为进料浓度 `C_A0`（mol/L）、进料温度 `T_in`（K）、冷却介质温度 `T_c`（K）和停留时间 `tau`（min）。一级反应速率 `r = k(T) C_A`，单位 mol/(L·min)。

物料与能量衡算为：

`dC_A/dt = (C_A0 - C_A)/tau - r`

`dT/dt = (T_in - T)/tau + [(-Delta H_r)/(rho Cp)] r + k_cool (T_c - T)`

`(-Delta H_r)/(rho Cp)` 的单位是 K·L/mol；`k_cool = UA/(rho Cp V)` 的单位是 min⁻¹。动力学基线采用 `k(T) = k_ref exp[-E_a/R (1/T - 1/T_ref)]`，并保证 `k_ref > 0`。实际装置若有显著物性变化、传质限制、压力/体积变化、多反应或热惯性，必须扩展相应机理，不能把它们默认交给网络掩盖。

**代入数字看量级：** 若 `C_A0=1.1`、`C_A=0.8 mol/L`、`tau=5 min`、`r=0.03 mol/(L·min)`，则进料带来的浓度变化为 `(1.1-0.8)/5=0.06 mol/(L·min)`，扣除反应后 `dC_A/dt=0.03 mol/(L·min)`。若 `T_in=350 K`、`T=360 K`、`T_c=300 K`，则进料项是 `-2 K/min`、冷却项是 `-4.8 K/min`；若反应热项为 `20×0.03=0.6 K/min`，总温度变化为 `-6.2 K/min`。每一步用 `DT=0.25 min` 时，Euler 近似温度下降 `1.55 K`。这些数字只帮助理解公式，不是现场参数。

In [ ]:
import os

# Keras 会在导入时决定后端，因此要先设置环境变量，再导入 keras。
# setdefault 表示“用户若已指定就尊重用户设置”；本示例只有在未设置时才默认用 TensorFlow。
os.environ.setdefault('KERAS_BACKEND', 'tensorflow')

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras import layers, ops

# 固定随机种子，让初始化权重、打乱批次等随机过程可复现，方便比较修改前后的结果。
keras.utils.set_random_seed(42)

# 所有时间统一用 min、温度用 K、浓度用 mol/L，避免速率方程中混用单位。
R = 8.314  # J/(mol K)，Arrhenius 方程中的气体常数
T_REF = 350.0  # K，作为速率方程和温度缩放的参考点
DT = 0.25  # min，每一步是 15 秒；步长越小通常越准确，但计算次数更多
N_STEPS = 40
HEAT_RELEASE = 20.0  # K L/mol；消耗 1 mol/L 的 A 对应约 20 K 的绝热温升尺度
K_COOL = 0.08  # min^-1；温差 10 K 时，冷却项贡献 0.8 K/min 的降温速率
# 40 个时间间隔需要 41 个状态点：t=0, 0.25, ..., 10 min。
TIME = np.arange(N_STEPS + 1) * DT

print(f'Keras {keras.__version__}; simulated horizon {TIME[-1]:.1f} min')

## 2. 生成可复现的动态运行轨迹

每条轨迹包含一次进料/温度/冷却/停留时间设定变化，以模拟操作条件变化。合成真值的活化能和弱温度相关速率扰动故意不同于后面的名义机理模型。测量噪声仅用于演示；现场数据还需要校准、时间对齐、缺失/异常检查和传感器延迟处理。

In [ ]:
def make_runs(n_runs, seed, tin_range=(335.0, 355.0), tc_range=(285.0, 305.0)):
    # 每次调用用独立随机数生成器：相同 seed 会得到相同数据，也不会改动全局随机状态。
    rng = np.random.default_rng(seed)
    # 每行是一条运行的初始工况：[进料浓度, 进料温度, 冷却温度, 停留时间]。
    # 例如 tau=5 min 表示理想混合釜内物料平均停留约 5 分钟。
    initial_conditions = np.column_stack([
        rng.uniform(0.8, 1.4, n_runs),
        rng.uniform(*tin_range, n_runs),
        rng.uniform(*tc_range, n_runs),
        rng.uniform(3.0, 8.0, n_runs),
    ])
    # controls 形状为 (运行数, 时间间隔数, 4)；先假设每条运行一直保持初始设定。
    controls = np.repeat(initial_conditions[:, None, :], N_STEPS, axis=1)

    # 每条运行在随机时刻改变设定值；所有设定值仍位于指定工况范围内。
    for run in range(n_runs):
        # 在第 10 到第 34 步间改变一次设定；DT=0.25 min，所以变化发生在 2.5–8.5 min。
        change = rng.integers(10, N_STEPS - 5)
        controls[run, change:, 0] = rng.uniform(0.8, 1.4)
        controls[run, change:, 1] = rng.uniform(*tin_range)
        controls[run, change:, 2] = rng.uniform(*tc_range)
        controls[run, change:, 3] = rng.uniform(3.0, 8.0)

    # 状态比控制多一个点：初始状态 t=0 加上每步更新后的状态。
    states = np.empty((n_runs, N_STEPS + 1, 2), dtype=np.float32)
    states[:, 0, 0] = controls[:, 0, 0]
    states[:, 0, 1] = controls[:, 0, 1]
    for step in range(N_STEPS):
        c_a0, t_in, t_c, tau = controls[:, step, :].T
        c_a, temp = states[:, step, :].T
        # 合成“真实”动力学使用 48 kJ/mol 活化能；模型之后只知道名义值 42 kJ/mol。
        # temp=350 K 时指数为 0，因此 k_true=0.030 min^-1。
        k_true = 0.030 * np.exp(-48_000 / R * (1 / temp - 1 / T_REF))
        k_true *= 1 + 0.10 * np.sin((temp - T_REF) / 10)
        # 一级反应：若 k=0.03 min^-1 且 C_A=1 mol/L，则 r=0.03 mol/(L min)。
        rate = k_true * c_a
        dc_a = (c_a0 - c_a) / tau - rate
        d_temp = (t_in - temp) / tau + HEAT_RELEASE * rate + K_COOL * (t_c - temp)
        # 显式 Euler：新值 = 旧值 + 步长 × 导数。max 防止离散步进产生负浓度。
        states[:, step + 1, 0] = np.maximum(c_a + DT * dc_a, 0)
        states[:, step + 1, 1] = temp + DT * d_temp

    # 给状态加测量噪声；例如温度噪声标准差 0.15 K，而不是改变控制输入。
    observed = states.copy()
    observed[:, :, 0] += rng.normal(0, 0.003, observed[:, :, 0].shape)
    observed[:, :, 1] += rng.normal(0, 0.15, observed[:, :, 1].shape)
    observed[:, :, 0] = np.maximum(observed[:, :, 0], 0)
    initial_state = observed[:, 0, :].copy()
    return controls.astype('float32'), initial_state, observed


u_train, s_train, y_train_raw = make_runs(160, seed=1)
u_val, s_val, y_val_raw = make_runs(32, seed=2)
u_test, s_test, y_test_raw = make_runs(40, seed=3)
u_extra, s_extra, y_extra_raw = make_runs(
    40, seed=4, tin_range=(355.0, 370.0), tc_range=(305.0, 320.0)
)

def scale_states(states):
    # 温度缩放到“相对 350 K 的 20 K 单位”：360 K -> 0.5，330 K -> -1。
    # 数值尺度接近 1 有助于神经网络训练；输出评估前会再还原成 K。
    return np.stack([states[:, :, 0], (states[:, :, 1] - T_REF) / 20], axis=-1).astype('float32')


y_train = scale_states(y_train_raw)
y_val = scale_states(y_val_raw)
y_test = scale_states(y_test_raw)
y_extra = scale_states(y_extra_raw)
print(f'按整条运行拆分：训练 {len(u_train)}，验证 {len(u_val)}，测试 {len(u_test)}；另有 {len(u_extra)} 条高温区间外推轨迹。')

## 3. 可微分机理层与灰箱修正

网络只学习有界的对数速率修正 `delta`，使 `k_eff = k_Arrhenius * exp(delta)` 始终为正；输出层零初始化，训练起点就是机理模型。两种模型使用相同物料和能量衡算、相同显式 Euler 时间步进。为突出结构差异，名义活化能固定为 42 kJ/mol（合成真值为 48 kJ/mol），拟合参考速率常数；现场应用应以测量精度、采样频率和数值稳定性选择积分器/步长并做收敛检查。

In [ ]:
class CSTRModel(keras.Model):
    def __init__(self, learn_correction):
        super().__init__()
        self.learn_correction = learn_correction
        # 直接学习 log(k_ref)，而不是 k_ref：无论权重取什么值，exp(log_k_ref) 都大于 0。
        # 以 log(0.025) 初始化，表示参考温度处的起始速率常数为 0.025 min^-1。
        self.log_k_ref = self.add_weight(
            name='log_k_ref', shape=(),
            initializer=keras.initializers.Constant(np.log(0.025)),
        )
        if learn_correction:
            # 这两层网络只学习机理速率的修正量，不直接替代物料和能量守恒方程。
            # 最后一层零初始化，使训练刚开始时 delta=0，灰箱先等同于机理模型。
            self.rate_net = keras.Sequential([
                layers.Dense(12, activation='tanh'),
                layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros'),
            ])

    def call(self, inputs):
        # controls: (批量, 40 步, 4 个操作量)；state0: (批量, 2 个初始状态)。
        controls, state0 = inputs
        c_a = state0[:, 0:1]
        temp = state0[:, 1:2]
        # 网络内部温度用 (T-350)/20 表示；例如 360 K 对应 0.5。
        # 先保存 t=0，因此最终预测会有 41 个状态点，而不是 40 个。
        states = [ops.concatenate([c_a, (temp - T_REF) / 20], axis=-1)]

        for step in range(N_STEPS):
            c_a0 = controls[:, step, 0:1]
            t_in = controls[:, step, 1:2]
            t_c = controls[:, step, 2:3]
            tau = controls[:, step, 3:4]
            # 350 K 时括号为 0，所以 k=k_ref；升温后速率按 Arrhenius 关系增大。
            k = ops.exp(self.log_k_ref - 42_000 / R * (1 / temp - 1 / T_REF))

            if self.learn_correction:
                features = ops.concatenate([
                    c_a, (temp - T_REF) / 20, (c_a0 - 1.1) / 0.3,
                    (t_in - T_REF) / 20, (t_c - 300) / 20, (tau - 5.5) / 2.5,
                ], axis=-1)
                # tanh 把修正限制在 [-0.25, 0.25]，防止网络把速率任意放大或压到接近零。
                # delta=0.1 时速率乘 exp(0.1)≈1.105，即增加约 10.5%；上下限约为 ×0.779 到 ×1.284。
                delta = 0.25 * ops.tanh(self.rate_net(features))
                k = k * ops.exp(delta)

            # 保留相同守恒式：灰箱仅把有效速率 k 改为 k*exp(delta)。
            rate = k * c_a
            dc_a = (c_a0 - c_a) / tau - rate
            d_temp = (t_in - temp) / tau + HEAT_RELEASE * rate + K_COOL * (t_c - temp)
            # 显式 Euler 更新；把浓度下限设为 0 是物理边界处理，不是训练标签。
            c_a = ops.maximum(c_a + DT * dc_a, 0)
            temp = temp + DT * d_temp
            states.append(ops.concatenate([c_a, (temp - T_REF) / 20], axis=-1))

        return ops.stack(states, axis=1)


def fit_physics_model(learn_correction):
    # 两个模型共用训练流程，仅用开关决定是否加入可学习的速率修正网络。
    model = CSTRModel(learn_correction)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss='mse')
    # 验证损失连续 8 轮不改善就停止，并恢复验证集表现最好的权重，减少过拟合。
    stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
    model.fit(
        (u_train, s_train), y_train,
        validation_data=((u_val, s_val), y_val),
        epochs=60, batch_size=16, callbacks=[stop], verbose=0,
    )
    return model


mechanistic = fit_physics_model(learn_correction=False)
greybox = fit_physics_model(learn_correction=True)
print('拟合参考速率常数 (min^-1):', float(ops.convert_to_numpy(ops.exp(mechanistic.log_k_ref))))

## 4. 纯数据模型对照

黑箱模型学习一步状态转移，再递归滚动整条轨迹。它和机理/灰箱模型使用相同的完整训练运行，不把测试运行的相邻时间点泄漏进训练。直接多步滚动时的小误差可能累积；模型也没有自动满足浓度非负、守恒或温度安全边界。

In [ ]:
def transition_features(state, controls):
    # 输入特征全部做量级缩放：例如 C_A0=1.4 时 (1.4-1.1)/0.3=1。
    # 这样神经网络各输入大致处于相近数值范围，优化器不必同时应付数量级差异很大的特征。
    return np.concatenate([
        state[..., 0:1], (state[..., 1:2] - T_REF) / 20,
        (controls[..., 0:1] - 1.1) / 0.3, (controls[..., 1:2] - T_REF) / 20,
        (controls[..., 2:3] - 300) / 20, (controls[..., 3:4] - 5.5) / 2.5,
    ], axis=-1).astype('float32')


def transition_data(controls, observed):
    # 用当前状态和当前操作量预测“下一步变化量”，而不是直接预测绝对状态。
    # 例：温度从 350.0 到 350.5 K，目标温度增量是 0.5/20=0.025；滚动时再乘回 20。
    features = transition_features(observed[:, :-1, :], controls)
    next_state = observed[:, 1:, :]
    current_state = observed[:, :-1, :]
    targets = np.stack([
        next_state[:, :, 0] - current_state[:, :, 0],
        (next_state[:, :, 1] - current_state[:, :, 1]) / 20,
    ], axis=-1).astype('float32')
    return features.reshape(-1, 6), targets.reshape(-1, 2)


# 160 条训练运行 × 40 步 = 6,400 个一步样本；验证集 32 × 40 = 1,280 个。
# 样本虽摊平成很多行，但训练/验证仍按整条运行切分，避免同一运行相邻点泄漏到两边。
x_step_train, y_step_train = transition_data(u_train, y_train_raw)
x_step_val, y_step_val = transition_data(u_val, y_val_raw)
blackbox = keras.Sequential([
    layers.Input(shape=(6,)),
    layers.Dense(32, activation='tanh'),
    layers.Dense(32, activation='tanh'),
    layers.Dense(2),
])
blackbox.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
# 同样依据独立验证运行选停止轮数；验证集不参与梯度更新。
stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
blackbox.fit(
    x_step_train, y_step_train,
    validation_data=(x_step_val, y_step_val),
    epochs=80, batch_size=256, callbacks=[stop], verbose=0,
)

def rollout_blackbox(model, controls, state0):
    # 一步网络必须递归调用 40 次，才能得到 10 分钟轨迹；每一步预测都成为下一步输入。
    # 因此单步很小的偏差可能逐渐累积，这也是不能只看一步验证损失的原因。
    state = state0.copy()
    states = [state]
    for step in range(N_STEPS):
        features = transition_features(state, controls[:, step, :])
        prediction = np.asarray(model(features, training=False))
        # 浓度增量直接加回 mol/L；温度增量先乘 20 还原为 K 再加回当前温度。
        state = np.column_stack([state[:, 0] + prediction[:, 0], state[:, 1] + 20 * prediction[:, 1]])
        states.append(state)
    return np.stack(states, axis=1)


def predict_physics(model, controls, state0):
    scaled = np.asarray(model.predict((controls, state0), verbose=0))
    return np.stack([scaled[:, :, 0], T_REF + 20 * scaled[:, :, 1]], axis=-1)


def evaluate_runs(controls, state0, observed):
    # 三类模型都从同一个首个观测状态开始，比较后续整条轨迹，而非单步拟合误差。
    predictions = {
        '机理': predict_physics(mechanistic, controls, state0),
        '灰箱': predict_physics(greybox, controls, state0),
        '黑箱': rollout_blackbox(blackbox, controls, state0),
    }
    rows = []
    for name, prediction in predictions.items():
        rows.append({
            '模型': name,
            # RMSE 是误差平方均值开根号；例如温度 RMSE=1 表示典型误差量级约 1 K。
            'C_A RMSE (mol/L)': np.sqrt(np.mean((prediction[:, :, 0] - observed[:, :, 0]) ** 2)),
            '温度 RMSE (K)': np.sqrt(np.mean((prediction[:, :, 1] - observed[:, :, 1]) ** 2)),
        })
    return pd.DataFrame(rows), predictions


test_scores, test_predictions = evaluate_runs(u_test, s_test, y_test_raw)
extra_scores, extra_predictions = evaluate_runs(u_extra, s_extra, y_extra_raw)
print('测试工况（训练范围内，按运行留出）')
display(test_scores.round(4))
print('高温工况（超出训练范围；外推仅作诊断，不作安全结论）')
display(extra_scores.round(4))

In [ ]:
# 画测试集第一条完整运行；run=0 是该测试批次中的索引，不是训练运行编号。
run = 0
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(TIME, y_test_raw[run, :, 0], 'k.', ms=3, label='测量/合成真值')
axes[1].plot(TIME, y_test_raw[run, :, 1], 'k.', ms=3, label='测量/合成真值')
for name, prediction in test_predictions.items():
    axes[0].plot(TIME, prediction[run, :, 0], label=name)
    axes[1].plot(TIME, prediction[run, :, 1], label=name)
axes[0].set_ylabel('C_A (mol/L)')
axes[1].set_ylabel('T (K)')
axes[1].set_xlabel('时间 (min)')
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()

## 5. 接入真实工厂数据前的检查

建议 CSV 一行对应一个等时间间隔采样点，至少包含下列列名和单位。`run_id` 必须区分独立运行；每个训练/验证/测试运行完整留出。对变工况过程，操作量按时间步提供，不能用整条轨迹的平均值替代。下面函数只做基础结构、有限值和时间顺序检查，不替代工艺工程师的数据审查、仪表校验、异常工况标注和安全审查。

In [ ]:
# 显式列名和单位能尽早发现错列、漏列；比默默用位置索引更容易审查。
PLANT_COLUMNS = [
    'run_id', 'time_min', 'c_a_mol_L', 'temperature_K', 'c_a0_mol_L',
    'feed_temperature_K', 'coolant_temperature_K', 'residence_time_min',
]

def load_plant_csv(path):
    # 逐项拒绝结构或数值异常，而不是悄悄填 0；错误数据应先按工艺规则处理。
    frame = pd.read_csv(path)
    missing = set(PLANT_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f'缺少必需列: {sorted(missing)}')
    numeric = [column for column in PLANT_COLUMNS if column != 'run_id']
    if frame[PLANT_COLUMNS].isna().any().any():
        raise ValueError('CSV 含缺失值；需先按工艺规则处理并记录处理方式。')
    if not np.isfinite(frame[numeric].to_numpy(dtype=float)).all():
        raise ValueError('CSV 数值列含 NaN 或无穷值。')
    # 排序后再检查时间，确保每条运行内部时间严格递增；重复时间会使 DT/导数计算失去定义。
    frame = frame.sort_values(['run_id', 'time_min']).reset_index(drop=True)
    if (frame.groupby('run_id')['time_min'].diff().dropna() <= 0).any():
        raise ValueError('每个 run_id 内时间必须严格递增，且不能有重复时间戳。')
    # 负浓度和非正停留时间违反当前模型假设，直接报错比继续训练并产生误导更安全。
    if (frame['c_a_mol_L'] < 0).any() or (frame['residence_time_min'] <= 0).any():
        raise ValueError('浓度应非负，停留时间应为正。')
    return frame


# 有真实数据后取消注释；本教程不假定或生成现场数据文件。
# plant = load_plant_csv('plant_runs.csv')
# 每条运行应整理成：controls (N_STEPS, 4)，列顺序为
# [c_a0_mol_L, feed_temperature_K, coolant_temperature_K, residence_time_min]；
# state0 为该运行首点 [c_a_mol_L, temperature_K]，target 为 N_STEPS+1 个观测状态。
# 先按工艺事件切分运行，再检查统一采样间隔 DT、操作量时序对齐和传感器延迟；
# 只有覆盖范围与状态数匹配的完整运行才能直接组成当前演示模型的批次张量。

## 6. 工业使用检查清单

- **单位与边界：** 核对温标、浓度/流量基准、停留时间、反应热、体积热容及传热系数；确认非负浓度、正停留时间和设备温压边界。
- **数据与验证：** 对齐化验值/在线仪表和控制系统时间戳；标记启动、停车、清洗、故障及切换工况；按独立批次/时间段留出测试，避免相邻采样泄漏。
- **可辨识性：** 进料、冷却和停留时间变化不足时，动力学、传热参数与神经网络修正可能互相补偿；检查灵敏度、参数区间和残差，而非只看训练误差。
- **外推与安全：** 分别报告训练域内和域外误差；检查能量/物料衡算残差、预测不确定性和物理边界。数据驱动模型不得绕过现有联锁、操作规程或独立安全审查。
- **生产部署：** 本示例使用显式 Euler 积分、已知常数热物性和合成数据。部署前需要用真实独立数据验证采样/积分误差、参数漂移、传感器故障和异常工况，并经工艺/控制/安全团队批准。